In [0]:
# import xgboost as xgb

# from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
import mlflow
import pyspark.sql.functions as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from mlflow.tracking import MlflowClient
from sklearn.ensemble import RandomForestRegressor
from mlflow.models.signature import infer_signature

In [0]:
catalog_name = "ct_oil_gas" 
gold_schema_name = "sc_gold"
model_name = f"{catalog_name}.{gold_schema_name}.daily_oil_gas_demand_xg_model"

RANDOM_STATE=  42

current_user ='abuhuzaifakurla@gmail.com'
xp_name = "exp_daily_oil_gas_demand"
run_name = 'daily_demand_xgb'
artifact_name = "daily_oil_gas_demand_model"

In [0]:
mlflow.set_registry_uri("databricks-uc")

In [0]:
print(f"Finding best run from {xp_name}_* and pushing new model version to {model_name}")
xp_path = f"/Users/{current_user}"

# mlflow.set_experiment(f"{xp_path}/{xp_name}")


experiment_id = mlflow.search_experiments(filter_string=f"name LIKE '{xp_path}/{xp_name}%'", order_by=["last_update_time DESC"])[0].experiment_id
print(experiment_id)

In [0]:
best_model = mlflow.search_runs(
    experiment_ids=experiment_id,
    order_by=["metrics.val_r2 DESC"],
    max_results=1,
    filter_string=f"status = 'FINISHED' and run_name= '{run_name}'"
)

best_model

In [0]:
print(f"Registering model to {model_name}") 

run_id = best_model.iloc[0]['run_id']

In [0]:
run_id

In [0]:

model_details = mlflow.register_model(
        model_uri=f"runs:/{run_id}/{artifact_name}",
        name=model_name,
)

In [0]:
model_details

In [0]:
client = MlflowClient()


client.update_registered_model(
  name=model_details.name,
  description="This model forecasts the demand of the next day based on product name,Destination_city."
)

In [0]:
best_score = best_model['metrics.val_r2'].values[0]
run_name = best_model['tags.mlflow.runName'].values[0]
version_desc = f"This model version has an R2 score metric of {round(best_score,4)*100}%. Follow the link to its training run for more details."


client.update_model_version(
    name=model_details.name,
    version=model_details.version,
    description=version_desc
)


client.set_model_version_tag(
    name=model_details.name,
    version= model_details.version,
    key='val_r2',
    value=f"{round(best_score,4)}"
)